# 🔒 3. PII Detection Middleware

LangChain provides built-in `PIIMiddleware` for detecting and handling **Personally Identifiable Information (PII)**.

PII middleware can detect sensitive information in user input and apply different strategies such as redacting, masking, hashing, or blocking it.

---

## 🧩 Supported PII Types

| Type | Example |
|---|---|
| `email` | `user@example.com` |
| `credit_card` | `5105-1051-0510-5100` |
| `ip` | `192.168.1.1` |
| `mac_address` | `00:1A:2B:3C:4D:5E` |
| `url` | `https://secret-site.com` |

---

## 🛠️ PII Handling Strategies

| Strategy | Result |
|---|---|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | `****-****-****-1234` |
| `hash` | `a8f5f167...` |
| `block` | Raises an exception |

---

## 🔄 How PII Middleware Works

```text
User Input
    ↓
PII Detection
    ↓
PII Found?
    ├── No  → ✅ Continue
    └── Yes
          ↓
    Apply Strategy
          ↓
    ┌───────────────┐
    │ Redact        │
    │ Mask          │
    │ Hash          │
    │ Block         │
    └───────────────┘

## 🔒 PII Detection Middleware Example

### Import Required Libraries

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

### Create a Dummy Customer Tool

In [4]:
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

### Create Agent with PII Middleware

In [6]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),

        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),

        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


### Test PII Redaction

In [8]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is iron.man@starkindustries.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

=== Agent Response ===
I’m happy to help you with your account. To protect your privacy and keep your information secure, I won’t store or process the card details you’ve shared.  

Could you let me know what you need assistance with (e.g., a recent transaction, a billing question, updating your account, etc.)? If you’d like me to look up your account, I can do that using the email address you provided—just confirm that it’s the correct email for the account you’re trying to access.  

Once I have a bit more context, I’ll be able to guide you through the next steps safely.


---

## 💡 Key Idea

> **PIIMiddleware detects sensitive information and applies a predefined strategy before the data reaches the model or agent.**

This helps prevent sensitive information such as emails, credit card numbers, and API keys from being unnecessarily exposed.

---

## 📝 Key Takeaways

- `PIIMiddleware` is a built-in LangChain middleware for handling PII.
- It supports different PII types such as **email, credit cards, IP addresses, MAC addresses, and URLs**.
- Common strategies include **redact, mask, hash, and block**.
- `apply_to_input=True` applies the middleware to incoming user input.
- Custom detectors can be provided using Regex patterns.
- PII middleware is useful for **privacy, security, and compliance**.